<a href="https://colab.research.google.com/github/Palomavergara/PFG/blob/main/TFG_PalomaVergara.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

In [ ]:
#Librerías necesarias
from google.colab import files
import cv2
import numpy as np
import math
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
import time

In [ ]:
# Video
print("Sube el vídeo que quieres analizar:")
video = files.upload()
nombreVideo = list(video.keys())[0]
print(f"\nVídeo cargado: {nombreVideo}")

# Menú
print("\n1. Exceso de velocidad")
print("2. Traspaso de línea continua")
print("3. Cambio de carril sin señalizar\n")

opcion = input("Elige una opción (1, 2 o 3): ")
while opcion not in ["1", "2", "3"]:
    opcion = input("Opción no válida. Elige 1, 2 o 3: ")

print(f"\nProcesando: {opcion}")

# Tracker de coches
class EuclideanDistTracker:
    def __init__(self):
        self.center_points = {}
        self.id_count = 0
        self.framesAusente = {}
        self.maxFramesAusente = 10  #Llevar un seguimiento si se pierde la señal

    def update(self, objects_rect):
        objects_bbs_ids = []
        #Calcula centros
        for rect in objects_rect:
            x, y, w, h = rect
            cx = (x + x + w) // 2
            cy = (y + y + h) // 2
            same_object_detected = False
            #Mira si ya existen
            for id, pt in self.center_points.items():
                dist = math.hypot(cx - pt[0], cy - pt[1])
                if dist < 50:
                    self.center_points[id] = (cx, cy)
                    self.framesAusente[id] = 0
                    objects_bbs_ids.append([x, y, w, h, id])
                    same_object_detected = True
                    break
            #Crea vehículos nuevos
            if not same_object_detected:
                self.center_points[self.id_count] = (cx, cy)
                self.framesAusente[self.id_count] = 0
                objects_bbs_ids.append([x, y, w, h, self.id_count])
                self.id_count += 1
        #Conteo de frames perdidos
        ids_vistos = [obj[4] for obj in objects_bbs_ids]
        for id in list(self.center_points.keys()):
            if id not in ids_vistos:
                self.framesAusente[id] = self.framesAusente.get(id, 0) + 1
        #Elimina coches ausentes
        new_center_points = {}
        nuevosFramesAusente = {}
        for id, pt in self.center_points.items():
            if self.framesAusente.get(id, 0) <= self.maxFramesAusente:
                new_center_points[id] = pt
                nuevosFramesAusente[id] = self.framesAusente.get(id, 0)

        self.center_points = new_center_points
        self.framesAusente = nuevosFramesAusente
        # Devuelve resultados
        return objects_bbs_ids


# EXCESO DE VELOCIDAD
if opcion == "1":

    # Inicializamos variables
    roiX1, roiY1 = 440, 657
    roiX2, roiY2 = 733, 863
    velocidadMax = 90
    tolerancia = 1.05  # Marca infracción a partir de 94.5
    estabilidad = 15  # Media de los últimos 15 frames

    modelo = YOLO("yolov8n.pt")
    tracker = EuclideanDistTracker()
    clasesVehiculo = [2, 3]

    posicionAnterior = {}
    desplazamientos = {}
    velocidades = {}
    vehiculosInfractores = set()  # Historial de las infracciones
    todosDesplazamientos = []
    factor = None  # Cambia cuando acaba la calibración

    cap = cv2.VideoCapture(nombreVideo)
    ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    alto  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    out   = cv2.VideoWriter("resultado_velocidad.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (ancho, alto))

    framesCalibración = int(fps * 4)
    print(f"Calibrando ...({framesCalibración} frames)")

    import time
    inicio = time.time()

    numFrame = 0
    #Recorre video frame a frame
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        numFrame += 1
        if numFrame % 50 == 0:
            print(f"Procesando frames: {numFrame}") #De 50 en 50
        #Dibujo ROI
        colorROI = (0, 255, 0) if factor else (0, 255, 255)
        cv2.rectangle(frame, (roiX1, roiY1), (roiX2, roiY2), colorROI, 2)
        # Calibrando
        if numFrame <= framesCalibración:
            cv2.putText(frame, f"Calibrando... ({numFrame}/{framesCalibración})",
                        (roiX1, roiY1 - 8), cv2.FONT_HERSHEY_SIMPLEX,
                        0.5, (0, 255, 255), 1)
        else:
            cv2.putText(frame, "ROI calibrado", (roiX1, roiY1 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

        if numFrame == framesCalibración + 1:
            if len(todosDesplazamientos) > 0:
                mediaGlobal = np.mean(todosDesplazamientos)
                factor = velocidadMax / mediaGlobal
                print(f"Calibración completada: {mediaGlobal:.2f} px/frame = 90 km/h")
                print(f"  Basada en {len(todosDesplazamientos)} mediciones")
            else:
                print("No hay datos para calibrar")

        # Detección de coches y motos de mínimo 20x20 frames con una confianza del 35%
        resultados = modelo(frame, verbose=False)[0]
        detecciones = []
        for box in resultados.boxes:
            clase = int(box.cls[0])
            if clase not in clasesVehiculo:
                continue
            if float(box.conf[0]) < 0.35:
                continue
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            w = x2 - x1
            h = y2 - y1
            if w < 20 or h < 20:
                continue
            detecciones.append([x1, y1, w, h])

        vehiculos = tracker.update(detecciones)

        for (x, y, w, h, vid) in vehiculos:
            cx = x + w // 2
            cy = y + h // 2
            dentroROI = (roiX1 < cx < roiX2 and roiY1 < cy < roiY2)

            if dentroROI: #Para calibrar
                if vid in posicionAnterior:
                    dy = cy - posicionAnterior[vid]
                    if 0 < dy < 50: #Movimientos realistas
                        if vid not in desplazamientos:
                            desplazamientos[vid] = []
                        desplazamientos[vid].append(dy)
                        if len(desplazamientos[vid]) > estabilidad: #Añade los 15 últimos al estudio
                            desplazamientos[vid].pop(0)
                        if numFrame <= framesCalibración: #Vector de calibración
                            todosDesplazamientos.append(dy)
                posicionAnterior[vid] = cy
            else:
                if vid in posicionAnterior:
                    del posicionAnterior[vid]

        for (x, y, w, h, vid) in vehiculos:
            #Centroides
            cx = x + w // 2
            cy = y + h // 2
            dentroROI = (roiX1 < cx < roiX2 and roiY1 < cy < roiY2)
            #Calculo de la velocidad si supera un historial de 5 frames
            if (factor and
                vid in desplazamientos and
                len(desplazamientos[vid]) >= 5):

                mediaVid = np.mean(desplazamientos[vid])
                velocidad = mediaVid * factor
                velocidades[vid] = velocidad

                if velocidad > velocidadMax * tolerancia:
                    vehiculosInfractores.add(vid)

            if vid in vehiculosInfractores:
                color = (0, 0, 255) #Rojo
                etiqueta = f"ID {vid} | {velocidades[vid]:.0f} km/h EXCESO" if vid in velocidades else f"ID {vid} EXCESO"
            else:
                color = (0, 200, 0) #Verde
                etiqueta = f"ID {vid} | {velocidades[vid]:.0f} km/h" if (vid in velocidades and dentroROI) else f"ID {vid}"

            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            cv2.putText(frame, etiqueta, (x, y - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

        out.write(frame)

    fin = time.time()
    fps_real = numFrame / (fin - inicio)
    print(f"\nFPS de procesamiento: {fps_real:.1f}")

    cap.release()
    out.release()

    print(f"\nProcesados {numFrame} frames en total.")

    if vehiculosInfractores:
        print(f"\nVehículos con exceso de velocidad: {sorted(vehiculosInfractores)}")
        print(f"\nTotal de infracciones: {len(vehiculosInfractores)} ")
        print("\nVelocidades registradas:")
        for vid in sorted(vehiculosInfractores):
            if vid in velocidades:
                print(f"ID {vid}: {velocidades[vid]:.1f} km/h")
    else:
        print("\nNo se detectaron infracciones.")

    files.download("resultado_velocidad.mp4")


# TRASPASO LÍNEA CONTINUA (DESVIACIÓN DEL CARRIL)
elif opcion == "2":
    #Se queda con los colores de las líneas previamente pintados (roja - continua y naranja - discontinua)
    def getMascaraRoja(frame):
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        rojo1 = cv2.inRange(hsv, np.array([0,   120, 70]), np.array([10,  255, 255]))
        rojo2 = cv2.inRange(hsv, np.array([170, 120, 70]), np.array([180, 255, 255]))
        return cv2.bitwise_or(rojo1, rojo2)
    # Comprueba si el punto inferior del coche ha pasado la línea roja
    def cruzaLinea(mascaraEstatica, cxInferior, cyInferior):
        margen = 5 #Margen de 5 píxeles que se tienen que pasar
        y1 = max(cyInferior - margen, 0)
        y2 = min(cyInferior + margen, mascaraEstatica.shape[0])
        x1 = max(cxInferior - margen, 0)
        x2 = min(cxInferior + margen, mascaraEstatica.shape[1])
        zona = mascaraEstatica[y1:y2, x1:x2]
        return cv2.countNonZero(zona) > 5

    cap = cv2.VideoCapture(nombreVideo)
    ret, primerFrame = cap.read()
    mascaraEstatica = getMascaraRoja(primerFrame)
    print("Líneas continuas detectadas en el primer frame.")
    cap.release()

    modelo = YOLO("yolov8n.pt")
    tracker = EuclideanDistTracker()
    vehiculosInfractores = set()
    clasesVehiculo = [2, 3]

    cap = cv2.VideoCapture(nombreVideo)
    ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    alto  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    out   = cv2.VideoWriter("resultado_lineas.mp4",cv2.VideoWriter_fourcc(*"mp4v"), fps, (ancho, alto))

    import time
    inicio = time.time()

    numFrame = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        numFrame += 1
        if numFrame % 50 == 0:
            print(f"Procesando frame {numFrame}...")

        resultados = modelo(frame, verbose=False)[0]
        detecciones = []
        for box in resultados.boxes:
            clase = int(box.cls[0])
            if clase not in clasesVehiculo:
                continue
            if float(box.conf[0]) < 0.35:
                continue
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            w = x2 - x1
            h = y2 - y1
            if w < 20 or h < 20:
                continue
            detecciones.append([x1, y1, w, h])

        vehiculos = tracker.update(detecciones)
        #Detecctamos infracción si el punto medio de abajo del box pasa la línea
        for (x, y, w, h, vid) in vehiculos:
            cxInferior = x + w // 2
            cyInferior = y + h

            if cruzaLinea(mascaraEstatica, cxInferior, cyInferior):
                vehiculosInfractores.add(vid)

            color    = (0, 0, 255) if vid in vehiculosInfractores else (0, 200, 0)
            etiqueta = f"ID {vid} INFRACTOR" if vid in vehiculosInfractores else f"ID {vid}"

            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            cv2.putText(frame, etiqueta, (x, y - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

        out.write(frame)

    fin = time.time()
    fps_real = numFrame / (fin - inicio)
    print(f"\nFPS de procesamiento: {fps_real:.1f}")

    cap.release()
    out.release()

    print(f"\nProcesados {numFrame} frames en total.")

    if vehiculosInfractores:
        print(f"\nVehículos infractores detectados: {sorted(vehiculosInfractores)}")
        print(f"\nTotal de infracciones: {len(vehiculosInfractores)}")
    else:
        print("\nNo se detectaron infracciones.")

    files.download("resultado_lineas.mp4")


# CAMBIO DE CARRIL SIN SEÑALIZAR
elif opcion == "3":
    #Detecta lineas rojas y naranjas previamente señalizadas
    def getMascarasEstaticas(frame):
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        rojo1 = cv2.inRange(hsv, np.array([0,   120, 70]), np.array([10,  255, 255]))
        rojo2 = cv2.inRange(hsv, np.array([170, 120, 70]), np.array([180, 255, 255]))
        mascaraRoja = cv2.bitwise_or(rojo1, rojo2)
        mascaraNaranjaSupelo = cv2.inRange(hsv, np.array([10, 120, 120]), np.array([25, 255, 255]))
        mascaraLineas = cv2.bitwise_or(mascaraRoja, mascaraNaranjaSupelo)
        return mascaraLineas, mascaraNaranjaSupelo
    #Detecta los intermitentes de los coches, descratando ruedas y techo
    def detectarIntermitente(historial, mascaraNaranjaSupelo, x, y, w, h, ancho, alto):
        margenLateral = max(int(w * 0.15), 8)
        xl1 = max(x - margenLateral, 0)
        xl2 = max(x + margenLateral, 0)
        xr1 = min(x + w - margenLateral, ancho)
        xr2 = min(x + w + margenLateral, ancho)
        y1z = max(y + int(h * 0.2), 0)
        y2z = min(y + int(h * 0.8), alto)

        zonaIzq = mascaraNaranjaActual[y1z:y2z, xl1:xl2]
        zonaDer = mascaraNaranjaActual[y1z:y2z, xr1:xr2]

        sueloIzq = mascaraNaranjaSupelo[y1z:y2z, xl1:xl2]
        sueloDer = mascaraNaranjaSupelo[y1z:y2z, xr1:xr2]

        intermitenteIzq = cv2.bitwise_and(zonaIzq, cv2.bitwise_not(sueloIzq))
        intermitenteDer = cv2.bitwise_and(zonaDer, cv2.bitwise_not(sueloDer))
        #Detecta el parpadeo de las luces guardando el historial
        pixeles = cv2.countNonZero(intermitenteIzq) + cv2.countNonZero(intermitenteDer)
        historial.append(pixeles)
        if len(historial) > 10:
            historial.pop(0)

        if len(historial) < 6:
            return False
        #Analoiza coches con un historial mínimo de 6 frames
        cambios = 0
        for i in range(1, len(historial[-6:])):
            if (historial[-6:][i] > 8) != (historial[-6:][i-1] > 8):
                cambios += 1
        return cambios >= 2
    # Detecta si cruza la línea como en el ejercicio anterior
    def cruzaLineaCarril(mascaraLineas, cxInferior, cyInferior):
        margen = 5
        y1 = max(cyInferior - margen, 0)
        y2 = min(cyInferior + margen, mascaraLineas.shape[0])
        x1 = max(cxInferior - margen, 0)
        x2 = min(cxInferior + margen, mascaraLineas.shape[1])
        zona = mascaraLineas[y1:y2, x1:x2]
        return cv2.countNonZero(zona) > 5

    cap = cv2.VideoCapture(nombreVideo)
    ret, primerFrame = cap.read()
    mascaraLineasEstatica, mascaraNaranjaEstatica = getMascarasEstaticas(primerFrame)
    print("Líneas de carril detectadas.")
    cap.release()

    modelo = YOLO("yolov8n.pt")
    tracker = EuclideanDistTracker()
    clasesVehiculo = [2, 3]

    historialIntermitente = {}
    intermitenteActivo = {}
    vehiculosInfractores = set()

    cap = cv2.VideoCapture(nombreVideo)
    ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    alto  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    out   = cv2.VideoWriter("resultado_intermitentes.mp4",cv2.VideoWriter_fourcc(*"mp4v"), fps, (ancho, alto))

    import time
    inicio = time.time()

    numFrame = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        numFrame += 1
        if numFrame % 50 == 0:
            print(f"Procesando frame {numFrame}...")

        hsvActual = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        mascaraNaranjaActual = cv2.inRange(hsvActual,
                                           np.array([10, 120, 120]),
                                           np.array([25, 255, 255]))

        resultados = modelo(frame, verbose=False)[0]
        detecciones = []
        for box in resultados.boxes:
            clase = int(box.cls[0])
            if clase not in clasesVehiculo:
                continue
            if float(box.conf[0]) < 0.35:
                continue
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            w = x2 - x1
            h = y2 - y1
            if w < 20 or h < 20:
                continue
            detecciones.append([x1, y1, w, h])

        vehiculos = tracker.update(detecciones)

        for (x, y, w, h, vid) in vehiculos:
            cxInferior = x + w // 2
            cyInferior = y + h

            if vid not in historialIntermitente:
                historialIntermitente[vid] = []

            intermitenteActivo[vid] = detectarIntermitente(
                historialIntermitente[vid],
                mascaraNaranjaEstatica,
                x, y, w, h, ancho, alto
            )

            if cruzaLineaCarril(mascaraLineasEstatica, cxInferior, cyInferior):
                if not intermitenteActivo[vid]:
                    vehiculosInfractores.add(vid)

            if vid in vehiculosInfractores:
                color = (0, 0, 255)
                etiqueta = f"ID {vid} SIN SEÑALIZAR"
            elif intermitenteActivo.get(vid, False):
                color = (0, 165, 255)
                etiqueta = f"ID {vid} INTERMITENTE"
            else:
                color = (0, 200, 0)
                etiqueta = f"ID {vid}"

            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            cv2.putText(frame, etiqueta, (x, y - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

        out.write(frame)

    fin = time.time()
    fps_real = numFrame / (fin - inicio)
    print(f"\nFPS de procesamiento: {fps_real:.1f}")

    cap.release()
    out.release()

    print(f"\nProcesados {numFrame} frames en total.")

    if vehiculosInfractores:
        print(f"\nVehículos que cambiaron de carril sin señalizar: {sorted(vehiculosInfractores)}")
        print(f"\nTotal infracciones: {len(vehiculosInfractores)}")
    else:
        print("\nNo se detectaron cambios de carril sin señalizar.")

    files.download("resultado_intermitentes.mp4")

Sube el vídeo que quieres analizar:


Saving videointermitentes.m4v to videointermitentes (1).m4v

Vídeo cargado: videointermitentes (1).m4v

1. Exceso de velocidad
2. Traspaso de línea continua
3. Cambio de carril sin señalizar

Elige una opción (1, 2 o 3): 3

Procesando: 3
Líneas de carril detectadas.
Procesando frame 50...
Procesando frame 100...
Procesando frame 150...
Procesando frame 200...
Procesando frame 250...
Procesando frame 300...
Procesando frame 350...
Procesando frame 400...
Procesando frame 450...

FPS de procesamiento: 56.8

Procesados 473 frames en total.

Vehículos que cambiaron de carril sin señalizar: [0, 2]

Total infracciones: 2


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>